# प्रतिमा निर्मिती अॅप्लिकेशन्स तयार करणे (OpenAI)

फक्त मजकूर निर्मितीपुरतीच LLMs मर्यादित नाहीत. तुम्ही मजकूर वर्णनांवरून प्रतिमा देखील तयार करू शकता. प्रतिमा ही एक माध्यम म्हणून MedTech, वास्तुकला, पर्यटन, गेम डेव्हलपमेंट, मार्केटिंग आणि इतर अनेक क्षेत्रांमध्ये उपयुक्त आहे. या धड्यात आपण OpenAI प्लॅटफॉर्मवरील आजच्या **GPT Image** मॉडेल्ससह काम करतो.

## शिकण्याची उद्दिष्टे

या धड्याच्या शेवटी तुम्ही करू शकणार आहात:

- प्रतिमा निर्मिती म्हणजे काय आणि ती कुठे उपयुक्त आहे हे समजावून सांगणे.
- `gpt-image` मॉडेल कुटुंब आणि ते पारंपरिक DALL·E मॉडेल्सपेक्षा कसे वेगळे आहे हे समजून घेणे.
- प्रतिमा निर्मिती अॅप्लिकेशन तयार करणे आणि प्रतिमा संपादित करणे.

## प्रतिमा निर्मिती म्हणजे काय?

प्रतिमा निर्मिती मॉडेल्स मजकूर प्रॉम्प्टवरून प्रतिमा तयार करतात. `gpt-image` सारखे आधुनिक मॉडेल्स प्रशिक्षणादरम्यान मजकूर आणि प्रतिमांमधील नाते शिकतात, आणि नंतर पुनरावृत्तीने यादृच्छिक आवाजाला (नॉइस) तुमच्या प्रॉम्प्टसाठी जुळणारी प्रतिमा बनवतात.

प्रतिमा मॉडेल्सचे दोन परिचित प्रकार आहेत:

- **`gpt-image` (OpenAI)** — सध्याचा पिढीचा मॉडेल जो या धड्यात वापरला जात आहे. हे मजकूर-ते-प्रतिमा निर्मिती आणि प्रतिमा संपादन (मास्कसह इनपेंटिंग) यांना समर्थन देते.
- **Midjourney** — एक लोकप्रिय तृतीय-पक्ष मॉडेल ज्याचे स्वतःचे सेवा आणि Discord-आधारित कार्यप्रवाह आहे.

> जुने OpenAI प्रतिमा मॉडेल्स — **DALL·E 2** आणि **DALL·E 3** — पारंपरिक आहेत. DALL·E 3 आता नवीन वितरणांसाठी उपलब्ध नाही, आणि `create_variation` सारखे फिचर्स फक्त DALL·E 2 मध्ये होते. नवीन अॅप्लिकेशन्ससाठी `gpt-image` मॉडेल्स वापरा.

> **महत्त्वाचे:** `gpt-image` मॉडेल्स तयार केलेली प्रतिमा **base64** (`b64_json`) स्वरूपात परत करतात, URL स्वरूपात नाही. तुमचा कोड base64 स्ट्रिंग डिकोड करून बाइट्समध्ये साठवतो — डाउनलोड करण्यासाठी कोणत्याही प्रतिमा URL नसते.


## तुमचे पहिले प्रतिमा निर्मिती अनुप्रयोग तयार करणे

तर प्रतिमा निर्मिती अनुप्रयोग तयार करण्यासाठी काय लागते? तुम्हाला खालील ग्रंथालयांची गरज आहे:

- **python-dotenv**, तुम्हाला ही ग्रंथालय तुमच्या रहस्यांचे कोडपासून दूर *.env* फाइलमध्ये ठेवण्यासाठी वापरणे अतिशय शिफारस केले आहे.
- **openai**, ह्या ग्रंथालयाचा वापर तुम्ही OpenAI API शी संवाद साधण्यासाठी कराल.
- **pillow**, Python मध्ये प्रतिमांवर काम करण्यासाठी.


1. खालील सामग्रीसह *.env* नावाची फाइल तयार करा:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. वरील लायब्ररींना *requirements.txt* नावाच्या फाईलमध्ये असे गोळा करा:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. नंतर, व्हर्च्युअल एन्व्हिरॉनमेंट तयार करा आणि लायब्ररी इंस्टॉल करा:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> विंडोजसाठी, आपले वर्चुअल एन्व्हायर्नमेंट तयार करण्यासाठी आणि सक्रिय करण्यासाठी खालील कमांड वापरा:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. *app.py* नावाच्या फाईलमध्ये खालील कोड जोडा:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # dotenv आयात करा
    dotenv.load_dotenv()

    # OpenAI वस्तू तयार करा (.env मधून OPENAI_API_KEY वाचतो)
    client = openai.OpenAI()


    try:
        # इमेज जनरेशन API वापरून प्रतिमा तयार करा
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # आपला प्रॉम्प्ट मजकूर येथे टाका
            size='1024x1024',
            n=1
        )
        # संग्रहित प्रतिमेसाठी निर्देशिका सेट करा
        image_dir = os.path.join(os.curdir, 'images')

        # जर निर्देशिका अस्तित्वात नसेल तर ती तयार करा
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # प्रतिमा मार्ग प्रारंभ करा (फाइलचा प्रकार png असावा याची नोंद घ्या)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image मॉडेल प्रतिमा base64 (b64_json) मध्ये परत करतात, URL मध्ये नाही
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # डिफॉल्ट प्रतिमा दर्शकात प्रतिमा दाखवा
        image = Image.open(image_path)
        image.show()

    # अपवाद पकडा
    except openai.BadRequestError as err:
        print(err)

    ```

या कोडचे स्पष्टीकरण करूया:

- प्रथम, आम्ही लागणाऱ्या लायब्ररी आयात करतो, ज्यात OpenAI लायब्ररी, dotenv लायब्ररी, base64 मॉड्यूल, आणि Pillow लायब्ररी यांचा समावेश आहे.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- त्यानंतर, आम्ही क्लायंट तयार करतो, जो तुमच्या ``.env`` मधून API की वाचतो.

    ```python
    # OpenAI ऑब्जेक्ट तयार करा
    client = openai.OpenAI()
    ```

- पुढे, आम्ही प्रतिमा तयार करतो:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # आपला प्रॉम्प्ट मजकूर येथे टाका
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` मॉडेल्स प्रतिमा **base64** स्ट्रिंग म्हणून परत देतात `data[0].b64_json` मध्ये. आम्ही ते बाइट्समध्ये डीकोड करतो आणि फाईलमध्ये लिहितो — डाउनलोडसाठी URL नाही.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- शेवटी, आम्ही प्रतिमा उघडतो आणि ती दर्शविण्यासाठी स्टँडर्ड इमेज व्ह्युअर वापरतो:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### प्रतिमा तयार करण्याबाबत अधिक तपशील

`images.generate` च्या पॅरामीटर्स पाहूया:

- **model**, वापरण्यात येणारे प्रतिमेचे मॉडेल (उदा. `gpt-image-1`).
- **prompt**, प्रतिमा तयार करण्यासाठी वापरलेला टेक्स्ट प्रॉम्प्ट. येथे तो "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils" आहे.
- **size**, तयार होणाऱ्या प्रतिमेचा आकार (उदा. `1024x1024`, `1536x1024`, `1024x1536`, किंवा `"auto"`).
- **n**, तयार होणाऱ्या प्रतिमा संख्या. येथे एक प्रतिमा तयार केली आहे.

> प्रतिमा मॉडेल्स `temperature` पॅरामीटर घेत नाहीत — हा टेक्स्ट-जनरेशन नियंत्रित करण्याचा पॅरामीटर आहे. विविधता वाढवण्यासाठी पुन्हा API कॉल करा; कमी करायची असल्यास तुमचा प्रॉम्प्ट अधिक विशिष्ट करा.

## प्रतिमा जनरेशनची अतिरिक्त क्षमता

तुम्ही पाहिलं कसं काही ओळी Python कोडने प्रतिमा तयार करता येते. `gpt-image` मॉडेल्स एक आधीची प्रतिमा **संपादित** देखील करू शकतात. प्रतिमा, ऐच्छिक **मास्क** (ज्यामुळे बदलायचे क्षेत्र ठरते), आणि प्रॉम्प्ट देऊन तुम्ही प्रतिमेचा एक भाग बदलू शकता — उदाहरणार्थ, आमच्या बनीला टोप बदलू शकता.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# संपादने ही base64 म्हणून देखील परत केली जातात
edited_image = base64.b64decode(response.data[0].b64_json)
```

मूळ प्रतिमेमध्ये फक्त ससा आहे; अंतिम प्रतिमेमध्ये सशावर टोप आहे.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
